# SV2 tuần 7 — packet từ file đến KV260
Chọn kernel Python PC có NumPy/Matplotlib. Chạy từng ô **Shift+Enter**; mọi ô chỉ đọc Git export, không SSH/build/inference. Giữ template không output; Save As bản làm việc trước khi lưu. 50 cặp sample/split lấy từ **5 ECG validation**, không phải accuracy test độc lập. Nghiệm thu số học DPU còn chờ review.

In [ ]:
from pathlib import Path
import csv, hashlib, html, json
from IPython.display import HTML, display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'results/week7/edge/status.json').is_file())
DATA = ROOT/'results/week7/edge'
EXPECTED_ARCHIVE = 'ca8bea529211e5890b9abf097c41a13632d9bc32f90b0b57b4a2ad12c8ad69bb'
for line in (DATA/'sha256sum_export.txt').read_text().splitlines():
    expected, name = line.split('  ', 1)
    p = DATA/name
    if not p.resolve().is_relative_to(DATA.resolve()): raise RuntimeError('Unsafe path')
    if hashlib.sha256(p.read_bytes()).hexdigest() != expected: raise RuntimeError('Checksum failed: '+name)
status = json.loads((DATA/'status.json').read_text())
provenance = json.loads((DATA/'provenance.json').read_text())
if provenance['archive_sha256'] != EXPECTED_ARCHIVE: raise RuntimeError('Unexpected source archive')
print('Nguồn:', DATA)
print('Run:', provenance['source_run_id'], '| Build UTC:', provenance['build_utc'])
print('Checksum Git export OK. Archive nguồn:', provenance['archive_sha256'])
print('Execution:', status['status'], '| Numeric:', status['numeric_acceptance'])
print('Số ECG khác nhau:', status['unique_ecg_samples'], '| Accuracy test:', status['independent_test_accuracy'], '| BLE:', status['ble_device_kv260'])


In [ ]:
def table(headers, rows):
    display(HTML('<table><tr>'+''.join('<th>'+html.escape(str(h))+'</th>' for h in headers)+'</tr>'+''.join('<tr>'+''.join('<td>'+html.escape(str(v))+'</td>' for v in row)+'</tr>' for row in rows)+'</table>'))
by_split = list(csv.DictReader((DATA/'by_split.csv').open()))
table(['Backend', 's', 'Vector', 'Khớp lớp', 'Max abs logits'], [[r['backend'],r['split'],r['vector_count'],r['class_matches'],r['max_abs_error']] for r in by_split])
print('Max abs so với tail FP32 trên tensor đã giải lượng tử. Khớp lớp không phải accuracy so nhãn test.')


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10,4))
for backend in ['reference','kv260']:
    rows = [r for r in by_split if r['backend']==backend]
    ax.plot([int(r['split']) for r in rows], [float(r['max_abs_error']) for r in rows], '.-', label=backend)
ax.set(xlabel='Logical split s',ylabel='Max abs logits (điểm số)',title='Sai khác logits theo split — validation pairs')
ax.set_xticks(range(11)); ax.grid(alpha=.3); ax.legend(); plt.show()
print('s10 identity: sai khác 0; không có model chạy. Biểu đồ không đặt gate INT8/DPU.')


In [ ]:
cases = json.loads((DATA/'cases.json').read_text())['rows']
SPLIT = 2  # Đổi số 0..10 để xem chi tiết; không chạy model.
selected = [r for r in cases if r['split']==SPLIT]
table(['Backend','Vector','Sample ID','Request ID','Max abs','Khớp lớp','Logits'], [[r['backend'],r['vector_index'],r['sample_id'],r['request_id'],r['max_abs_error'],r['class_match'],r['logits']] for r in selected])
socket_runs = json.loads((DATA/'socket.json').read_text())['runs']
table(['Socket run','Phạm vi','Kết quả'], [[r['run_id'],r['scope'],r['rows']] for r in socket_runs])
print('Socket là HOST s10 identity smoke riêng; không chứng minh BLE hoặc TCP/DPU end-to-end.')
